# Application test: simple cleaning and preprocessing

This notebook prepares `application_test.csv` using the same rules as the application-train notebook. Each code cell does one small task.

- Test has no `TARGET`: we do not create a target or split this file.
- Keep every application and keep its ID separately.
- Use the saved train preprocessor for missing values, encoding, and scaling. Never fit it on test.
- This is the **application-only** version. Historical features will be joined later, before fitting a new combined preprocessor on training rows only.

Run the cells from top to bottom. The final save cell writes new test outputs; the raw CSV, train files, and historical notebooks stay unchanged.

## 1. Imports

`pandas` reads tables, `numpy` handles numeric missing values, and `Path` handles folders. `joblib` loads the preprocessing object already saved by the train notebook.

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
from scipy import sparse
from IPython.display import display

## 2. Find the project and file paths

Start in the current folder and look in its parents for the raw-data folder. This avoids a personal drive or username in the notebook.

In [2]:
project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data/raw/home-credit-default-risk").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside the project folder.")

data_folder = project_root / "data/raw/home-credit-default-risk"
preprocessor_path = project_root / "artifacts/preprocessors/application/preprocessor.joblib"
manifest_path = project_root / "artifacts/metadata/application/manifest.json"

print("Data folder:", data_folder)

Data folder: e:\Depi Notes Data Engineering\DEPI project R5\ProJect\data\raw\home-credit-default-risk


## 3. Read application_test.csv

One row is one current application. There is no outcome column because these are applications for which we will later predict risk.

In [3]:
df_test = pd.read_csv(data_folder / "application_test.csv")

print("Original shape:", df_test.shape)
display(df_test.head())

Original shape: (48744, 121)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Check IDs, columns, and missing values

`assert` stops the notebook if a required condition is false. Repeated application IDs must be investigated rather than silently deleted.

In [4]:
print("Duplicate rows:", df_test.duplicated().sum())
print("Duplicate IDs:", df_test["SK_ID_CURR"].duplicated().sum())
print("Missing IDs:", df_test["SK_ID_CURR"].isna().sum())

assert "TARGET" not in df_test.columns
assert df_test["SK_ID_CURR"].notna().all()
assert df_test["SK_ID_CURR"].is_unique
assert df_test.columns.is_unique

df_test.info()

Duplicate rows: 0
Duplicate IDs: 0
Missing IDs: 0
<class 'pandas.DataFrame'>
RangeIndex: 48744 entries, 0 to 48743
Columns: 121 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(40), str(16)
memory usage: 51.2 MB


In [5]:
missing_report = pd.DataFrame({
    "missing_count": df_test.isna().sum(),
    "missing_percent": df_test.isna().mean() * 100,
})

display(missing_report.sort_values("missing_percent", ascending=False).head(15).round(2))

,missing_count,missing_percent
COMMONAREA_AVG,33495,68.72
COMMONAREA_MEDI,33495,68.72
COMMONAREA_MODE,33495,68.72
NONLIVINGAPARTMENTS_AVG,33347,68.41
NONLIVINGAPARTMENTS_MEDI,33347,68.41
NONLIVINGAPARTMENTS_MODE,33347,68.41
FONDKAPREMONT_MODE,32797,67.28
LIVINGAPARTMENTS_MEDI,32780,67.25
LIVINGAPARTMENTS_AVG,32780,67.25
LIVINGAPARTMENTS_MODE,32780,67.25


High missingness is not an automatic reason to remove a column. We reuse the column choices learned from training. Numeric gaps will use training medians, not test medians.

## 5. Clean the employment placeholder

`365243` is a placeholder, not a real employment duration. Keep a flag showing where it appeared, then replace it with `NaN` (missing). Use the unchanged original table to build the flag, so rerunning this cell is safe.

Keep `XNA`, `Unknown`, and extreme amounts as recorded, matching train. Do not delete applicants or introduce new cleaning rules only for test.

In [6]:
df_test_clean = df_test.copy()

employment_placeholder = df_test["DAYS_EMPLOYED"].eq(365243)
df_test_clean["DAYS_EMPLOYED_PLACEHOLDER"] = employment_placeholder.astype(int)
df_test_clean["DAYS_EMPLOYED"] = df_test_clean["DAYS_EMPLOYED"].replace(365243, np.nan)

print("Employment placeholders flagged:", employment_placeholder.sum())
assert not df_test_clean["DAYS_EMPLOYED"].eq(365243).any()
assert df_test_clean["DAYS_EMPLOYED_PLACEHOLDER"].sum() == employment_placeholder.sum()
assert len(df_test_clean) == len(df_test)

Employment placeholders flagged: 9274


## 6. Create the same features as train

Negative day offsets describe the past. Dividing their negative by `365.25` gives approximate years. The car-age flag is 1 when car age was missing.

In [7]:
df_test_features = df_test_clean.copy()

df_test_features["AGE_YEARS"] = -df_test_features["DAYS_BIRTH"] / 365.25
df_test_features["EMPLOYMENT_YEARS"] = -df_test_features["DAYS_EMPLOYED"] / 365.25
df_test_features["OWN_CAR_AGE_MISSING"] = df_test_features["OWN_CAR_AGE"].isna().astype(int)

### Ratios

- Credit / income: how large the requested credit is compared with income.
- Annuity / income: how large the periodic installment is compared with income. This is not a monthly burden percentage unless the time units match.
- Credit / goods price: how the credit amount compares with the goods price.

A denominator must be positive and finite. Otherwise keep the ratio missing; the saved training imputer will handle it.

In [8]:
income = df_test_features["AMT_INCOME_TOTAL"]
valid_income = income.where((income > 0) & np.isfinite(income))

goods_price = df_test_features["AMT_GOODS_PRICE"]
valid_goods_price = goods_price.where((goods_price > 0) & np.isfinite(goods_price))

df_test_features["CREDIT_INCOME_RATIO"] = df_test_features["AMT_CREDIT"] / valid_income
df_test_features["ANNUITY_INCOME_RATIO"] = df_test_features["AMT_ANNUITY"] / valid_income
df_test_features["CREDIT_GOODS_RATIO"] = df_test_features["AMT_CREDIT"] / valid_goods_price

ratio_columns = ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_GOODS_RATIO"]
df_test_features[ratio_columns] = df_test_features[ratio_columns].replace([np.inf, -np.inf], np.nan)

The year columns contain the same information as the original day offsets, so remove those two original columns, just as we did in train.

In [9]:
df_test_features = df_test_features.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED"])

new_columns = ["AGE_YEARS", "EMPLOYMENT_YEARS", "OWN_CAR_AGE_MISSING"] + ratio_columns
display(df_test_features[["SK_ID_CURR"] + new_columns].head())
print("Featured table:", df_test_features.shape)

,SK_ID_CURR,AGE_YEARS,EMPLOYMENT_YEARS,OWN_CAR_AGE_MISSING,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_GOODS_RATIO
0,100001,52.678987,6.376454,1,4.213333,0.152300,1.2640
1,100005,49.456537,12.235455,1,2.250182,0.175455,1.2376
2,100013,54.861054,12.205339,0,3.275378,0.344578,1.0528
3,100028,38.264203,5.108830,1,5.000000,0.155614,1.0000
4,100038,35.701574,5.998631,0,3.475000,0.178150,1.0000


Featured table: (48744, 126)


## 7. Separate IDs from model inputs

The ID helps match predictions back to applications. It is not a measurement for the model. There is no `y_test` because the test outcome is not provided.

In [10]:
test_ids = df_test_features["SK_ID_CURR"].copy()
X_test = df_test_features.drop(columns="SK_ID_CURR")

print("Test inputs:", X_test.shape)
assert "SK_ID_CURR" not in X_test.columns
assert "TARGET" not in X_test.columns

Test inputs: (48744, 125)


## 8. Load training's saved column choices and preprocessor

Run the application-train notebook first if these files are missing. The saved preprocessor contains the training medians, numeric scales, and text categories. We load it; we do not create or fit a new one.

In [11]:
assert preprocessor_path.is_file(), "Run the application-train notebook first: saved preprocessor is missing."
assert manifest_path.is_file(), "Run the application-train notebook first: manifest is missing."

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
preprocessor = joblib.load(preprocessor_path)

retained_columns = manifest["retained_features"]
categorical_columns = manifest["categorical_features"]

print("Input columns selected by train:", len(retained_columns))
print("Prepared columns from train:", len(manifest["transformed_features"]))

Input columns selected by train: 125
Prepared columns from train: 255


## 9. Match training columns and their order

Select the same columns in the same order as training. A missing required column is an error to investigate; do not silently create it with zeros. Normalize real text gaps to `np.nan`, matching the train notebook.

In [12]:
missing_columns = [column for column in retained_columns if column not in X_test.columns]
assert not missing_columns, f"Required test columns are missing: {missing_columns}"
assert retained_columns == preprocessor.feature_names_in_.tolist()

selected_test = X_test[retained_columns].copy()

for column in categorical_columns:
    values = selected_test[column].astype(object)
    selected_test[column] = values.where(pd.notna(values), np.nan)

assert selected_test.columns.tolist() == retained_columns
print("Aligned test inputs:", selected_test.shape)

Aligned test inputs: (48744, 125)


## 10. Fill missing values, encode, and scale

One `transform` applies all three steps using the saved training rules:

- Numeric missing values: training medians, then training scaling.
- Text missing values: `__MISSING__`, then one-hot encoding.
- New text categories: handled by the saved encoder's `handle_unknown="ignore"`; no new columns are added.

**Use `transform`, never `fit` or `fit_transform`, on test.** The standard scikit-learn preprocessor is simply the saved collection of these rules; no additional project scripts are needed.

In [13]:
X_test_prepared = preprocessor.transform(selected_test)
feature_names = preprocessor.get_feature_names_out()

print("Prepared test:", X_test_prepared.shape)
display(pd.Series(feature_names[:15], name="example_feature"))

Prepared test: (48744, 255)


0                   numeric__CNT_CHILDREN
1               numeric__AMT_INCOME_TOTAL
2                     numeric__AMT_CREDIT
3                    numeric__AMT_ANNUITY
4                numeric__AMT_GOODS_PRICE
5     numeric__REGION_POPULATION_RELATIVE
6              numeric__DAYS_REGISTRATION
7                numeric__DAYS_ID_PUBLISH
8                    numeric__OWN_CAR_AGE
9                     numeric__FLAG_MOBIL
10                numeric__FLAG_EMP_PHONE
11               numeric__FLAG_WORK_PHONE
12              numeric__FLAG_CONT_MOBILE
13                    numeric__FLAG_PHONE
14                    numeric__FLAG_EMAIL
Name: example_feature, dtype: str

## 11. Check the result

Keep all rows and their order. Prepared test must have exactly the same feature names as prepared train, with no missing or infinite numeric values. Sparse output stores mostly nonzero values to save memory; do not turn the whole matrix into a dense table.

In [14]:
assert X_test_prepared.shape[0] == len(df_test)
assert X_test_prepared.shape[1] == len(manifest["transformed_features"])
assert feature_names.tolist() == manifest["transformed_features"]
assert test_ids.tolist() == df_test["SK_ID_CURR"].tolist()
assert selected_test.index.equals(test_ids.index)

prepared_values = X_test_prepared.data if sparse.issparse(X_test_prepared) else X_test_prepared
assert np.isfinite(prepared_values).all()

print("Checks passed: all applications kept, IDs aligned, train/test features match, output is finite.")

Checks passed: all applications kept, IDs aligned, train/test features match, output is finite.


In [15]:
# Only make five rows dense for an easy-to-read preview.
preview = X_test_prepared[:5]
if sparse.issparse(preview):
    preview = preview.toarray()

display(pd.DataFrame(preview, columns=feature_names).iloc[:, :10])

,numeric__CNT_CHILDREN,numeric__AMT_INCOME_TOTAL,numeric__AMT_CREDIT,numeric__AMT_ANNUITY,numeric__AMT_GOODS_PRICE,numeric__REGION_POPULATION_RELATIVE,numeric__DAYS_REGISTRATION,numeric__DAYS_ID_PUBLISH,numeric__OWN_CAR_AGE,numeric__FLAG_MOBIL
0,-0.578749,-0.130727,-0.075829,-0.450912,-0.239752,-0.145164,-0.051188,1.446789,-0.146470,0.002016
1,-0.578749,-0.269745,-0.935056,-0.670634,-0.970592,1.079452,-1.171331,0.909087,-0.146470,0.002016
2,-0.578749,0.129931,0.158733,2.938518,0.247475,-0.127021,0.798566,-0.337373,-0.710442,0.002016
3,2.200964,0.564360,2.422650,1.508927,2.805416,0.399993,0.848217,-0.804795,-0.146470,0.002016
4,0.811107,0.043045,0.064962,0.341515,0.235294,-0.782555,0.280769,-0.840598,0.840480,0.002016


## 12. Save test results when you are ready

This is the only cell that writes output files. It creates new test files and does not replace train outputs.

| Output | Purpose |
|---|---|
| `application_test_cleaned.joblib` | Cleaned application data before feature engineering. |
| `application_test_featured.joblib` | Features before filling, encoding, or scaling; use this table for the later historical join. |
| `X_test.npz` | Prepared numeric application-only matrix for the baseline model. |
| `test_ids.csv` | Application IDs in exactly the same row order as the matrix. |

For the combined version, join historical features to **both** unencoded train and test feature tables by `SK_ID_CURR`. Reuse the existing training/validation membership and fit a new combined preprocessor on the training portion only. Do not merge history onto the already-scaled matrix.

In [16]:
interim_output = project_root / "data/interim/application"
data_output = project_root / "data/processed/application"

interim_output.mkdir(parents=True, exist_ok=True)
data_output.mkdir(parents=True, exist_ok=True)

joblib.dump(df_test_clean, interim_output / "application_test_cleaned.joblib", compress=3)
joblib.dump(df_test_features, interim_output / "application_test_featured.joblib", compress=3)

saved_matrix = sparse.csr_matrix(X_test_prepared, dtype=np.float32)
assert np.isfinite(saved_matrix.data).all()
sparse.save_npz(data_output / "X_test.npz", saved_matrix)
test_ids.to_frame().to_csv(data_output / "test_ids.csv", index=False)

print("Saved the application-only test results.")

Saved the application-only test results.


## What is finished?

The test notebook now defines the same employment cleaning, year features, missing flag, ratios, column selection, missing-value filling, encoding, and scaling used for the application-only train baseline. No prediction model is trained here.

The combined application + historical version and model evaluation are separate remaining tasks. Prepared data alone does not establish prediction quality.